Tuple = (Actor, Action, Object, Condition)

Metrics:

Exact Match F1

Span Match F1 (token overlap)

Semantic Match F1 (SentenceTransformer)

And simple error analysis:

Missing tuples (LLM missed)

Extra tuples (LLM hallucinated)

No component-level metrics.

**Install dependencies**

In [ ]:
!pip install pandas openpyxl sentence-transformers

**Imports and semantic model**

In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# semantic similarity model
model = SentenceTransformer("all-MiniLM-L6-v2")

Parse annotation into tuples

Each argument unit becomes:

(actor, action, object, condition) **bold text**

In [ ]:
def parse_annotation(text):
    """
    Convert annotation text into list of tuples:
    (actor, action, object, condition)
    """

    tuples = []

    if not isinstance(text, str):
        return tuples

    current = {"actor":"", "action":"", "object":"", "condition":""}

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        # remove numbering like "1."
        line = re.sub(r"^\d+\.\s*", "", line)

        m = re.match(r"(Actor|Action|Object|Condition):\s*(.*)", line, re.I)

        if m:

            comp = m.group(1).lower()
            span = m.group(2).strip()

            # new tuple starts when a new Actor appears
            if comp == "actor":
                if any(current.values()):
                    tuples.append(current)
                    current = {"actor":"", "action":"", "object":"", "condition":""}

            current[comp] = span

    if any(current.values()):
        tuples.append(current)

    return tuples

**Matching functions**

In [ ]:
def exact_match(a,b):
    return a.strip().lower() == b.strip().lower()

In [ ]:
import re
import nltk
from collections import Counter
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

lemmatizer = WordNetLemmatizer()

def token_overlap_old(a: str, b: str) -> float:
    """
    Compute normalized token overlap (token-level F1) between two text spans.
    Counts duplicate tokens. Returns a value between 0.0 and 1.0.

    Text is normalized by:
        - Lowercasing
        - Removing punctuation
        - Stripping extra whitespace
        - Lemmatization
    """
    # Normalize text
    def normalize(text):
        text = text.lower()
        text = re.sub(r'[\W_]+', '', text) # More robust punctuation removal
        text = text.strip()
        return text

    def tokenize(text):
        tokens = nltk.word_tokenize(text)
        # Lemmatize each token for robustness
        tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
        return Counter(tokens)

    a_tokens = tokenize(normalize(a))
    b_tokens = tokenize(normalize(b))

    if not a_tokens or not b_tokens:
        return 0.0

    overlap_count = sum((a_tokens & b_tokens).values())
    normalized_overlap = (2 * overlap_count) / (sum(a_tokens.values()) + sum(b_tokens.values()))
    return normalized_overlap

In [ ]:
import re

def token_overlap(a: str, b: str, threshold=0.5):
    """
    Relaxed span match based on token overlap.
    Returns True if overlap proportion exceeds threshold.

    - Converts text to lowercase
    - Removes punctuation
    - Computes overlap as |tokens_a ∩ tokens_b| / max(|tokens_a|, |tokens_b|)
    """
    # Normalize
    def norm(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = text.strip()
        return text

    a_norm = norm(a)
    b_norm = norm(b)

    tokens_a = set(a_norm.split())
    tokens_b = set(b_norm.split())

    if not tokens_a or not tokens_b:
        return False

    overlap = tokens_a & tokens_b
    score = len(overlap) / max(len(tokens_a), len(tokens_b))

    return score >= threshold

In [ ]:
def tuple_match_threshold(g, p, match_func, threshold=0.7):
    """
    Boolean tuple match using weighted Action + Object scoring.

    - Action (main verb) is mandatory.
    - Object is optional but contributes to score.
    - Returns True if weighted score >= threshold, False otherwise.

    Args:
        g (dict): gold tuple
        p (dict): predicted tuple
        match_func (callable): Exact / TokenOverlap / Semantic match function
        threshold (float): weighted score threshold for TP

    Returns:
        bool: True if tuple considered a match
    """
    # --- Extract / preprocess fields ---
    g_action =  extract_main_verb(g.get("action", ""))
    p_action = extract_main_verb(p.get("action", ""))
    #p_action = p.get("action", "")

    # Action must match
    if not match_func(g_action, p_action):
        return False

    # Object preprocessing
    g_obj = preprocess_text(g.get("object", ""))
    p_obj = preprocess_text(p.get("object", ""))

    action_score = 1.0  # already matched
    object_score = 1.0 if match_func(g_obj, p_obj) else 0.0

    weighted_score = 0.7 * action_score + 0.3 * object_score

    return weighted_score >= threshold

In [ ]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

STOPWORDS = {
    "a","an","the","for","of","to","in","on","at",
    "with","by","and","or","is","are","be"
}

def semantic_match(a, b, threshold=0.7):

    def normalize(text):

        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)

        tokens = [
            w for w in text.split()
            if w not in STOPWORDS
        ]

        return " ".join(tokens)

    a_norm = normalize(a)
    b_norm = normalize(b)

    emb = model.encode([a_norm, b_norm])

    emb = np.array([v/np.linalg.norm(v) for v in emb])

    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]

    return sim >= threshold

Tuple matching

A tuple matches if all components match.

In [ ]:
def span_match(a, b):
    return a.strip().lower() == b.strip().lower()

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Actor normalization mapping
ACTOR_MAP = {
    "you": "facility",
    "the producer": "facility",
    "the owner": "facility",
    "facility": "facility"
}

def normalize_actor(text):
    """Normalize actor aliases to a canonical form."""
    t = text.lower().strip()
    return ACTOR_MAP.get(t, t)



def extract_main_verb(action_text):
    """
    Extract the main verb from a sentence using spacy.
    Returns lemmatized root verb, or original text if no verb found.
    """
    doc = nlp(action_text)
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ == "VERB":
            return token.lemma_
    return action_text

def preprocess_text(text: str) -> str:
    """
    Normalize text for component-level matching:
    - Lowercase
    - Remove punctuation
    - Strip extra whitespace
    """
    if not text:
        return ""

    text = text.lower()                     # lowercase
    text = re.sub(r'[^\w\s]', '', text)    # remove punctuation
    text = text.strip()                     # remove leading/trailing whitespace
    return text

def tuple_match_priority_llm(g, p, match_func, must_match=["action"], actor_optional=True):
    """
    Match tuples with:
      - Action (main verb) as must-match
      - Actor optional (normalized)
      - Object/Condition optional (preprocessed)

    Notes:
    - Action: extract main verb before comparison
    - Object/Condition: preprocessed for relaxed matching
    - Actor: normalized, optional
    """

    # --- Must-match components ---
    for c in must_match:
        g_val = g.get(c, "")
        p_val = p.get(c, "")

        if c == "action":
            g_val = extract_main_verb(g_val)
            p_val = extract_main_verb(p_val)

        if not match_func(g_val, p_val):
            return False  # must-match failed → tuple fails

    # --- Optional Actor ---
    if actor_optional:
        g_val = normalize_actor(g.get("actor", ""))
        p_val = normalize_actor(p.get("actor", ""))
        _ = match_func(g_val, p_val)  # optional, does not block tuple

    # --- Optional Object and Condition ---
    for c in ["object", "condition"]:
        g_val = preprocess_text(g.get(c, ""))
        p_val = preprocess_text(p.get(c, ""))
        _ = match_func(g_val, p_val)  # optional, does not block tuple

    return True

**Tuple-level evaluation**

Definitions:

TP: predicted tuple matches a gold tuple

FP: predicted tuple with no matching gold tuple

FN: gold tuple not predicted

In [ ]:
def tuple_eval(gold_data, pred_data, match_func):
    """
    Tuple-level evaluation using LLM-aware tuple match:
      - Action: main verb must-match
      - Actor: normalized, optional
      - Object/Condition: optional
    Computes precision, recall, F1.
    """
    tp, fp, fn = 0, 0, 0

    for ex in gold_data:
        gold = gold_data[ex]
        pred = pred_data.get(ex, [])
        matched = set()

        for p_tuple in pred:
            found = False
            for i, g_tuple in enumerate(gold):
                if i in matched:
                    continue
                # Use LLM-aware tuple match here
                if tuple_match_threshold(g_tuple, p_tuple, match_func):
                    tp += 1
                    matched.add(i)
                    found = True
                    break
            if not found:
                fp += 1

        # Remaining gold tuples are false negatives
        fn += len(gold) - len(matched)

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return precision, recall, f1

**Simple error analysis**
Tracks:

Missing tuples

Extra tuples

In [ ]:
def error_analysis_extended(gold_data, pred_data, match_func, threshold=0.7):
    """
    Returns:
        missing: list of (example_id, gold_tuple, missing_components)
        extra: list of (example_id, pred_tuple, extra_components)
    missing_components / extra_components: list of fields that did not match
    """
    missing = []
    extra = []

    for ex in gold_data:
        gold = gold_data[ex]
        pred = pred_data.get(ex, [])
        matched = set()

        # --- Check missing gold tuples ---
        for g in gold:
            found = False
            for j, p in enumerate(pred):
                if tuple_match_threshold(g, p, match_func, threshold=threshold):
                    matched.add(j)
                    found = True
                    break
            if not found:
                # Identify which components are failing
                comp_miss = []
                for c in ["action", "object", "actor", "condition"]:
                    g_val = g.get(c, "")
                    p_vals = [p.get(c, "") for p in pred]
                    if not any(match_func(g_val, pv) for pv in p_vals):
                        comp_miss.append(c)
                missing.append((ex, g, comp_miss))

        # --- Check extra predicted tuples ---
        for j, p in enumerate(pred):
            if j not in matched:
                # Identify which components are extra (not matching any gold)
                comp_extra = []
                for c in ["action", "object", "actor", "condition"]:
                    p_val = p.get(c, "")
                    g_vals = [g.get(c, "") for g in gold]
                    if not any(match_func(p_val, gv) for gv in g_vals):
                        comp_extra.append(c)
                extra.append((ex, p, comp_extra))

    return missing, extra

Component Level

In [ ]:
# =========================
# Component-level evaluation for AAOC
# =========================

def component_eval(gold_data, pred_data, match_func):
    """
    Computes precision, recall, F1 per tuple component:
      - Action: main verb extraction
      - Actor: normalized, optional
      - Object/Condition: optional
    """
    metrics = {c: [0,0,0] for c in ["actor","action","object","condition"]}

    for ex in gold_data:
        gold = gold_data[ex]
        pred = pred_data.get(ex, [])

        for comp in metrics:
            gold_spans = [g[comp] for g in gold]
            pred_spans = [p[comp] for p in pred]

            # Normalize / extract main verb
            # Normalize / extract main verb
            '''
            if comp == "actor":
                gold_spans = [normalize_actor(x) for x in gold_spans]
                pred_spans = [normalize_actor(x) for x in pred_spans]
            elif comp == "action":
                gold_spans = [extract_main_verb(x) for x in gold_spans]
                pred_spans = [extract_main_verb(x) for x in pred_spans]
            else:  # Object / Condition
                gold_spans = [preprocess_text(x) for x in gold_spans]
                pred_spans = [preprocess_text(x) for x in pred_spans]

            '''

            matched = set()
            # Count TP/FP
            for p_span in pred_spans:
                found = False
                for i, g_span in enumerate(gold_spans):
                    if i in matched:
                        continue
                    if match_func(g_span, p_span):
                        metrics[comp][0] += 1  # TP
                        matched.add(i)
                        found = True
                        break
                if not found:
                    metrics[comp][1] += 1  # FP
            # Count FN
            metrics[comp][2] += len(gold_spans) - len(matched)

    # Compute precision, recall, F1
    results = {}
    for c, (tp, fp, fn) in metrics.items():
        precision = tp/(tp+fp) if tp+fp>0 else 0
        recall = tp/(tp+fn) if tp+fn>0 else 0
        f1 = 2*precision*recall/(precision+recall) if precision+recall>0 else 0
        results[c] = (precision, recall, f1)

    return results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from openpyxl import load_workbook

# ========================================
# Excel configuration
# ========================================
EXCEL_PATH = '/content/drive/MyDrive/ArgMining 2026/Legal Text Extraction_All.xlsx'


SHEET_NAME = "New Text"  # or specify sheet name

GOLD_ID_COLUMN = "B"
GOLD_COLUMN = "E"

#LLM_ID_COLUMN = "B"
#LLM_PRED_COLUMN = "F"# gpt-oss
#LLM_PRED_COLUMN = "G"# phi
#LLM_PRED_COLUMN = "H"# AdaptLaw

# column indices (1-based) matching Task1 structure
# B: tagID, D: Text, E: Final Gold Label, F: gpt-oss-safeguard:20b,
# G: phi3.5, H: AdaptLaw  (adjust if other columns are present)
COL_TAG_IDX = 1
COL_TEXT_IDX = 3
COL_GOLD_IDX = 4
COL_GPT_IDX = 5
COL_PHI_IDX = 6
COL_ADL_IDX = 7

# derive column names later from dataframe

CATEGORY_ORDER = ["only shall", "only must", "both shall and must"]
INCLUDE_EXACT_PLOT = True

print("Task 2 config loaded")


In [ ]:
from pathlib import Path
from typing import Dict, List

def load_dataset(path: str, sheet_name=SHEET_NAME) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Excel not found: {path}")
    df = pd.read_excel(p, sheet_name=sheet_name)
    # ensure we have at least through AdaptLaw if needed
    if len(df.columns) <= COL_ADL_IDX:
        raise ValueError("Excel does not have expected columns through AdaptLaw (H).")
    cols = list(df.columns)
    selected = df[[
        cols[COL_TAG_IDX],
        cols[COL_TEXT_IDX],
        cols[COL_GOLD_IDX],
        cols[COL_GPT_IDX],
        cols[COL_PHI_IDX],
        cols[COL_ADL_IDX],
    ]].copy()
    selected.columns = [
        "tag_id",
        "text",
        "gold_label",
        "llm_gpt_oss_safeguard_20b",
        "llm_phi35",
        "llm_AdaptLaw",
    ]
    selected["tag_id"] = selected["tag_id"].astype(str).str.strip()
    selected = selected[selected["tag_id"].notna() & (selected["tag_id"] != "")]
    selected = selected.drop_duplicates(subset=["tag_id"], keep="first")
    selected = selected.reset_index(drop=True)
    return selected

def build_annotation_maps(df: pd.DataFrame, llm_col: str):
    gold_data: Dict[str, List[Dict[str, str]]] = {}
    pred_data: Dict[str, List[Dict[str, str]]] = {} # Initialize pred_data
    text_by_id: Dict[str, str] = {}

    for _, row in df.iterrows():
        tid = str(row["tag_id"]).strip()
        text_by_id[tid] = str(row.get("text", ""))
        gold_data[tid] = parse_annotation(row.get("gold_label", ""))
        pred_data[tid] = parse_annotation(row.get(llm_col, ""))

    return gold_data, pred_data, text_by_id # Return pred_data instead of pred_data_processed

In [ ]:
def detect_start_row(worksheet, input_column: str) -> int:
    first_value = worksheet[f"{input_column}1"].value
    if isinstance(first_value, str) and first_value.strip().lower() in {
        "excerpt", "excerpts", "text", "input", "excerpt_id", "tagid", "tag_id"
    }:
        return 2
    return 1


In [ ]:
# =========================
# Task 2 helpers
# =========================
def bucket_excerpt(text):
    txt = str(text or "").lower()
    has_shall = bool(re.search(r"\bshall\b", txt))
    has_must = bool(re.search(r"\bmust\b", txt))

    if has_shall and has_must:
        return "both shall and must"
    if has_shall:
        return "only shall"
    if has_must:
        return "only must"
    return "other"




def subset_by_category(gold_data, pred_data, text_by_id, category):
    ids = [ex_id for ex_id, txt in text_by_id.items() if bucket_excerpt(txt) == category]
    gold_sub = {ex_id: gold_data.get(ex_id, []) for ex_id in ids}
    pred_sub = {ex_id: pred_data.get(ex_id, []) for ex_id in ids}
    return gold_sub, pred_sub


def fraction_missed_extra(gold_data, pred_data, match_func):
    missing, extra = error_analysis_extended(gold_data, pred_data, match_func)
    gold_total = sum(len(v) for v in gold_data.values())
    pred_total = sum(len(pred_data.get(k, [])) for k in gold_data.keys())

    fraction_missed = len(missing) / gold_total if gold_total else 0.0
    fraction_extra = len(extra) / pred_total if pred_total else 0.0
    return fraction_missed, fraction_extra, gold_total, pred_total


def grouped_bar_plot(df, value_col, title, ylabel):
    data = df.copy()
    data = data[data["category"].isin(CATEGORY_ORDER)]
    if data.empty:
        print(f"No data for plot: {title}")
        return

    pivot = (
        data.pivot(index="category", columns="model", values=value_col)
        .reindex(CATEGORY_ORDER)
        .fillna(0.0)
    )

    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(title)
    ax.set_xlabel("Excerpt category")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
    ax.legend(title="LLM", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
f1_rows = []
err_rows = []

matchers = {
   # "Exact": exact_match,
    "Span": token_overlap,
    "Semantic": semantic_match
}

# load unified dataset once
df = load_dataset(EXCEL_PATH, SHEET_NAME)
# mapping from readable model name to column in dataframe
model_columns = {
    "gpt-oss-safeguard:20b": "llm_gpt_oss_safeguard_20b",
    "phi3.5": "llm_phi35",
    "AdaptLaw": "llm_AdaptLaw",
}

In [ ]:
def tuple_score(g, p, match_func):
    """
    Compute a simple similarity score between a gold tuple g and predicted tuple p.
    Uses Action + Object as main scoring fields.
    Returns a score in [0,1].
    """
    # Action: main verb extraction
    g_action = extract_main_verb(g.get("action", ""))
    p_action = extract_main_verb(p.get("action", ""))

    action_score = 1.0 if match_func(g_action, p_action) else 0.0

    # Object: preprocessed, token/semantic match
    g_obj = preprocess_text(g.get("object", ""))
    p_obj = preprocess_text(p.get("object", ""))
    object_score = 1.0 if match_func(g_obj, p_obj) else 0.0

    # Weighted sum (can adjust weights)
    return 0.7 * action_score + 0.3 * object_score


def filter_llm_output(pred_tuples, gold_tuples, match_func):
    """
    Select predicted tuples that best match gold tuples.
    Returns up to len(gold_tuples) predicted tuples.
    """
    scores = []
    for p in pred_tuples:
        # score against all gold tuples, take max
        s = max([tuple_score(g, p, match_func) for g in gold_tuples]) if gold_tuples else 0
        scores.append(s)

    # Sort predicted tuples by score descending
    sorted_preds = [p for _, p in sorted(zip(scores, pred_tuples), key=lambda x: x[0], reverse=True)]

    # Take top N tuples
    n = min(len(gold_tuples)*2, len(pred_tuples))
    return sorted_preds[:n]


In [ ]:
FACILITY_TERMS = [
    "person", "persons", "owner", "operator", "manufacturer",
    "entity", "establishment", "facility", "producer", "you"
]

def normalize_actor_pred(actor_text: str) -> str:
    """
    Normalize LLM-predicted actors:
    Map any mention of person/owner/manufacturer/entity to 'Facility'.
    """
    if not actor_text:
        return ""

    text = actor_text.lower().strip()

    if any(term in text for term in FACILITY_TERMS):
        return "Facility"

    return actor_text.strip()

In [ ]:
def bucket_excerpt(text):
    txt = str(text or "").lower()
    has_shall = bool(re.search(r"\bshall\b", txt))
    has_must = bool(re.search(r"\bmust\b", txt))

    if has_shall and has_must:
        return "both shall and must"
    if has_shall:
        return "only shall"
    if has_must:
        return "only must"
    return "other"




def subset_by_category(gold_data, pred_data, text_by_id, category):
    ids = [ex_id for ex_id, txt in text_by_id.items() if bucket_excerpt(txt) == category]
    gold_sub = {ex_id: gold_data.get(ex_id, []) for ex_id in ids}
    pred_sub = {ex_id: pred_data.get(ex_id, []) for ex_id in ids}
    return gold_sub, pred_sub

In [ ]:
import matplotlib.pyplot as plt

def grouped_bar_plot(df, value_col, title, ylabel):
    data = df.copy()
    data = data[data["category"].isin(CATEGORY_ORDER)]
    if data.empty:
        print(f"No data for plot: {title}")
        return

    pivot = (
        data.pivot(index="category", columns="model", values=value_col)
        .reindex(CATEGORY_ORDER)
        .fillna(0.0)
    )

    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(title)
    ax.set_xlabel("Excerpt category")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
    ax.legend(title="LLM", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

Run Eval

In [ ]:
# =========================
# Run Task 2 + Task 2a (combined) - Optimized
# =========================
import pandas as pd

f1_rows = []
err_rows = []

matchers = {
    # "Exact": exact_match,
    "Span": token_overlap,
    "Semantic": semantic_match
}

# Load unified dataset once
df = load_dataset(EXCEL_PATH, SHEET_NAME)

# Mapping from readable model name to column in dataframe
model_columns = {
    "GPT-OSS-20B": "llm_gpt_oss_safeguard_20b",
    "Phi-3 Mini-3.8B": "llm_phi35",
    "AdaptLaw": "llm_AdaptLaw",
}

# Precompute SBERT embeddings for semantic match (optional)
sbert_model = SentenceTransformer("all-miniLM-L6-v2")

for model_name, llm_col in model_columns.items():
    # --- Build annotation maps ---
    gold_data, pred_data, text_by_id = build_annotation_maps(df, llm_col)

    # --- Preprocess predicted tuples once ---
    pred_data_processed = {}
    for ex in gold_data:
        processed_pred = []
        for t in pred_data.get(ex, []):
            processed_pred.append({
                "action": extract_main_verb(t.get("action", "")),
                "actor": normalize_actor_pred(t.get("actor", "")),
                "object": preprocess_text(t.get("object", "")),
                "condition": preprocess_text(t.get("condition", ""))
            })
        pred_data_processed[ex] = processed_pred

    # Optional: precompute embeddings for semantic match
    pred_embeddings = {}
    gold_embeddings = {}
    for ex in gold_data:
        gold_embeddings[ex] = []
        for g in gold_data[ex]:
            gold_embeddings[ex].append({
                "action": sbert_model.encode(g.get("action", "")),
                "actor": sbert_model.encode(g.get("actor", "")),
                "object": sbert_model.encode(g.get("object", "")),
                "condition": sbert_model.encode(g.get("condition", ""))
            })
        pred_embeddings[ex] = []
        for p in pred_data_processed[ex]:
            pred_embeddings[ex].append({
                "action": sbert_model.encode(p.get("action", "")),
                "actor": sbert_model.encode(p.get("actor", "")),
                "object": sbert_model.encode(p.get("object", "")),
                "condition": sbert_model.encode(p.get("condition", ""))
            })

    for category in CATEGORY_ORDER:
        gold_sub, pred_sub = subset_by_category(gold_data, pred_data_processed, text_by_id, category)
        if not gold_sub:
            continue

        # Task 2: tuple F1 by matcher
        for match_name, func in matchers.items():
            p, r, f1 = tuple_eval(gold_sub, pred_sub, func)
            f1_rows.append({
                "model": model_name,
                "category": category,
                "match": match_name,
                "precision": p,
                "recall": r,
                "tuple_f1": f1
            })

        # Task 2a: fraction missed / extra
        frac_missed_span, _, gold_total, _ = fraction_missed_extra(gold_sub, pred_sub, token_overlap)
        _, frac_extra_sem, _, pred_total = fraction_missed_extra(gold_sub, pred_sub, semantic_match)

        err_rows.append({
            "model": model_name,
            "category": category,
            "fraction_missed_span": frac_missed_span,
            "fraction_extra_semantic": frac_extra_sem,
            "gold_tuples": gold_total,
            "pred_tuples": pred_total
        })

# Convert results to DataFrame
f1_df = pd.DataFrame(f1_rows)
err_df = pd.DataFrame(err_rows)

# Display results
print("Task 2 - Tuple F1 by model/category/matcher")
display(f1_df.sort_values(["match", "category", "model"]))

print("Task 2a - Fraction missed/extra by model/category")
display(err_df.sort_values(["category", "model"]))

# -------------------------
# Plotting
# -------------------------
grouped_bar_plot(
    f1_df[f1_df["match"] == "Span"],
    value_col="tuple_f1",
    title="Tuple F1 (Span Match) by category and LLM",
    ylabel="Tuple F1"
)

grouped_bar_plot(
    f1_df[f1_df["match"] == "Semantic"],
    value_col="tuple_f1",
    title="Tuple F1 (Semantic Match) by category and LLM",
    ylabel="Tuple F1"
)

grouped_bar_plot(
    err_df,
    value_col="fraction_missed_span",
    title="Fraction missed (Span Match) by category and LLM",
    ylabel="Fraction missed"
)

grouped_bar_plot(
    err_df,
    value_col="fraction_extra_semantic",
    title="Fraction extra (Semantic Match) by category and LLM",
    ylabel="Fraction extra"
)